# Task 3.3d: Hợp nhất và nghiệm thu kết quả trích xuất đồ thị

Hợp nhất output của các lượt chạy Task 3.3 thành bộ 9.077 hàm với Graph A (AST) và Graph B (AST + CFG + DATA_DEP), sau đó nghiệm thu độ bao phủ.

**Chuẩn bị:** Add Input toàn bộ output của các lượt chạy (R1–R7, Run 2–4, phân đoạn 2 và 3 cũ). Notebook không cần Internet ngoài việc clone mã nguồn.

## 1. Chuẩn bị mã nguồn

In [ ]:
import csv
import json
import re
import subprocess
import sys
from pathlib import Path

GIT_BRANCH = "feature/3_3_graph-extract"
REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "software-bug-detection-using-graphs"
EXPECTED_TOTAL = 9077

if not (WORKING.is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle.")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "origin", GIT_BRANCH], cwd=REPO_DIR, check=True)


## 2. Tìm thư mục output đầu vào

Notebook nhận cả thư mục `output_*` lẫn tệp `output_*.zip` trong `/kaggle/input`; tệp zip được giải nén vào `/kaggle/working/unzipped`.

Thứ tự gộp: các thư mục `output_batch_*` cũ (không ghim phiên bản Joern) trước, các thư mục `output_range_*` mới sau và theo chỉ số tăng dần. Thư mục gộp sau sẽ ghi đè thư mục gộp trước nếu trùng mẫu.

In [ ]:
import zipfile

UNZIP_DIR = WORKING / "unzipped"
for z in sorted(Path("/kaggle/input").rglob("output_*.zip")):
    target = UNZIP_DIR / z.stem
    if not target.exists():
        with zipfile.ZipFile(z) as zf:
            zf.extractall(target)
        print("Đã giải nén:", z.name)

def sort_key(path: Path):
    m = re.match(r"output_range_(\d+)_(\d+)$", path.name)
    if m:
        return (1, int(m.group(1)), 0)
    m = re.match(r"output_batch_(\d+)", path.name)
    return (0, int(m.group(1)) if m else 0, 0)

by_name = {}
for root in (Path("/kaggle/input"), UNZIP_DIR):
    for p in root.rglob("output_*"):
        if p.is_dir() and re.match(r"output_(range|batch)_\d", p.name) and (p / "graph_a").is_dir():
            by_name.setdefault(p.name, p)
batch_dirs = sorted(by_name.values(), key=sort_key)

print(f"{'Thư mục':<60}{'graph_a':>9}{'graph_b':>9}{'manifest':>10}")
for d in batch_dirs:
    n_a = len(list((d / "graph_a").glob("*.json")))
    n_b = len(list((d / "graph_b").glob("*.json")))
    n_m = sum(sum(1 for _ in open(f, encoding="utf-8")) - 1 for f in d.glob("manifest_*.csv"))
    print(f"{str(d):<60}{n_a:>9}{n_b:>9}{n_m:>10}")

# Các đoạn mới cần có đủ. Hai phân đoạn cũ (3000-5699 và 6000-8436) nằm ở output_batch_*.
EXPECTED_RANGES = [
    (0, 250), (250, 500), (500, 750), (750, 1000), (1000, 2000), (2000, 3000),
    (5700, 6000), (8437, 8657), (8657, 8680), (8680, 8683), (8683, 8867), (8867, 9077),
]
found = {tuple(map(int, re.findall(r"\d+", d.name))[:2]) for d in batch_dirs if d.name.startswith("output_range_")}
missing = [r for r in EXPECTED_RANGES if r not in found]
print("\nĐoạn mới còn thiếu trong input:", missing or "không có")
print("Số thư mục output_batch_* cũ:", sum(d.name.startswith("output_batch_") for d in batch_dirs), "(cần 2)")


## 3. Hợp nhất và lập báo cáo

In [ ]:
MERGED_DIR = WORKING / "merged_3_3" / "subset_9077"
REPORT_PATH = WORKING / "reports" / "graph" / "3_3_extraction.md"
FAILED_PATH = REPORT_PATH.with_name("3_3_failed_samples.csv")

cmd = [
    sys.executable, "scripts/merge_and_report_3_3.py",
    "--batch-dirs", *map(str, batch_dirs),
    "--output-dir", str(MERGED_DIR),
    "--report-out", str(REPORT_PATH),
    "--failed-out", str(FAILED_PATH),
    "--joern-version", "4.0.654 (các lần chạy đầu không ghim phiên bản)",
]
result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
print(result.stdout)
print(result.stderr, file=sys.stderr)
result.check_returncode()


## 4. Nghiệm thu

Đạt khi mỗi graph có đủ 9.077 tệp, không mẫu nào chưa hoàn tất, và tổng thành công cộng bị loại bằng 9.077.

In [ ]:
official_ids = [r["sample_id"] for r in csv.DictReader(open(REPO_DIR / "data/splits/diversevul_experiment_manifest.csv", encoding="utf-8"))]
assert len(official_ids) == EXPECTED_TOTAL, len(official_ids)

summary = {}
for kind in ("graph_a", "graph_b"):
    statuses = {}
    unreadable = []
    for sid in official_ids:
        f = MERGED_DIR / kind / f"{sid}.json"
        try:
            statuses[sid] = json.loads(f.read_text(encoding="utf-8")).get("status", "UNKNOWN")
        except (OSError, ValueError):
            unreadable.append(sid)
    ok = sum(s in {"SUCCESS", "OVERSIZED_GRAPH"} for s in statuses.values())
    summary[kind] = {"có tệp": len(statuses), "thiếu hoặc hỏng": len(unreadable),
                     "thành công": ok, "bị loại": len(statuses) - ok}
    from collections import Counter
    print(kind, summary[kind])
    print("  ", dict(Counter(statuses.values())))

failed_rows = list(csv.DictReader(open(FAILED_PATH, encoding="utf-8")))
quarantined = sorted(r["sample_id"] for r in failed_rows if r["status"] == "QUARANTINED")
print("\nMẫu cách ly:", quarantined)

for kind, s in summary.items():
    assert s["thiếu hoặc hỏng"] == 0, f"{kind}: còn mẫu thiếu hoặc hỏng"
    assert s["thành công"] + s["bị loại"] == EXPECTED_TOTAL, f"{kind}: tổng không bằng {EXPECTED_TOTAL}"
assert set(quarantined) == {"row-1355", "row-17527", "row-17523"}, quarantined
print("\nNGHIỆM THU ĐẠT: bao phủ 100% cho Graph A và Graph B.")


## 5. Đóng gói

- `graphs_subset_9077.zip`: toàn bộ đồ thị, lưu thành Kaggle Dataset private và sao lưu lên Drive. Không commit lên git.
- `reports/graph/3_3_extraction.md` và `3_3_failed_samples.csv`: tải về và commit vào nhánh.

In [ ]:
import shutil
archive = shutil.make_archive(str(WORKING / "graphs_subset_9077"), "zip", MERGED_DIR.parent, MERGED_DIR.name)
print("Đã tạo:", archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")
shutil.copy(REPORT_PATH, WORKING / "3_3_extraction.md")
shutil.copy(FAILED_PATH, WORKING / "3_3_failed_samples.csv")
print("Tải về: 3_3_extraction.md, 3_3_failed_samples.csv")
